In [1]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchaudio
import torchaudio.transforms as AT

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

import wandb
from kaggle_secrets import UserSecretsClient

In [ ]:
try:
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("W&B login successful!")
except Exception as e:
    print(f"W&B login via secrets failed ({e}), trying anonymous...")
    wandb.login(anonymous="allow")

In [4]:
AUDIO_DIR = "/kaggle/input/datasets/sreekaranreddy2005/dlgenai-proj-audio-classification/messy-mashup-augmented/audio"
CSV_PATH = "/kaggle/input/datasets/sreekaranreddy2005/dlgenai-proj-audio-classification/messy-mashup-augmented/metadata.csv"
OUTPUT_DIR = "/kaggle/working"
WANDB_PROJECT = "24f2000010-t12026"

In [5]:
# Audio params
SAMPLE_RATE = 22050
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
AUDIO_DURATION = 5 

# Model params
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 35
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 10
SEED = 42
N_FOLDS = 5
TRAIN_FOLD = 0

# CRNN architecture params
CNN_FEATURES = 128       
RNN_HIDDEN = 128         
RNN_LAYERS = 2           
RNN_DROPOUT = 0.3

In [6]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [7]:
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for g, i in GENRE_TO_IDX.items()}

In [ ]:
print("Loading metadata...")
df = pd.read_csv(CSV_PATH)

if 'filepath' in df.columns:
    df['filename'] = df['filepath'].apply(lambda x: os.path.basename(x))
elif 'image' in df.columns:
    df['filename'] = df['image'].str.replace('.png', '.wav')

if 'genre' in df.columns:
    df['labels'] = df['genre']
elif 'labels' not in df.columns:
    df.columns = ['filepath', 'labels']
    df['filename'] = df['filepath'].apply(lambda x: os.path.basename(x))

df['song_group'] = df['filename'].apply(
    lambda x: '_'.join(os.path.splitext(x)[0].split('_')[:2])
)
df['label_idx'] = df['labels'].map(GENRE_TO_IDX)

existing_files = set(os.listdir(AUDIO_DIR))
df = df[df['filename'].isin(existing_files)].reset_index(drop=True)

print(f"Total samples: {len(df)}")
print(f"Unique songs:  {df['song_group'].nunique()}")
print(f"Genre distribution:\n{df['labels'].value_counts()}\n")

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold_idx, (train_idx, val_idx) in enumerate(
    sgkf.split(df, df['label_idx'], groups=df['song_group'])
):
    if fold_idx == TRAIN_FOLD:
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)
        break

print(f"Fold {TRAIN_FOLD}: Train={len(train_df)}, Val={len(val_df)}")


In [ ]:
class AudioDataset(Dataset):
    def __init__(self, dataframe, audio_dir, sample_rate=22050,
                 n_mels=128, n_fft=2048, hop_length=512,
                 duration=3, augment=False):
        self.df = dataframe
        self.audio_dir = audio_dir
        self.sample_rate = sample_rate
        self.duration = duration
        self.target_length = sample_rate * duration
        self.augment = augment

        # Mel spectrogram transform
        self.mel_transform = AT.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            power=2.0
        )
        self.amplitude_to_db = AT.AmplitudeToDB(stype='power', top_db=80)

    def __len__(self):
        return len(self.df)
    def _load_audio(self, path):
        """Load and normalize audio to fixed length."""
        waveform, sr = torchaudio.load(path)

        
        if sr != self.sample_rate:
            resampler = AT.Resample(sr, self.sample_rate)
            waveform = resampler(waveform)

        # Pad or trim to target length
        if waveform.shape[1] < self.target_length:
            padding = self.target_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))
        else:
            waveform = waveform[:, :self.target_length]

        return waveform

    def _augment_waveform(self, waveform):
        """Simple waveform augmentations."""
        # Random volume change
        if random.random() < 0.5:
            gain = random.uniform(0.7, 1.3)
            waveform = waveform * gain

        # Random noise injection
        if random.random() < 0.3:
            noise = torch.randn_like(waveform) * 0.005
            waveform = waveform + noise

        # Random time shift
        if random.random() < 0.3:
            shift = random.randint(-2000, 2000)
            waveform = torch.roll(waveform, shift, dims=-1)

        return waveform

    def _spec_augment(self, mel_spec):
        """SpecAugment: mask frequency and time bands."""
        _, n_mels, time_steps = mel_spec.shape

        # Frequency masking (2 masks)
        for _ in range(2):
            f = random.randint(1, 15)
            f0 = random.randint(0, max(0, n_mels - f))
            mel_spec[:, f0:f0+f, :] = mel_spec.min()

        # Time masking (2 masks)
        for _ in range(2):
            t = random.randint(1, 20)
            t0 = random.randint(0, max(0, time_steps - t))
            mel_spec[:, :, t0:t0+t] = mel_spec.min()

        return mel_spec

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        label = row['label_idx']

        # Load audio
        waveform = self._load_audio(audio_path)

        # Augment waveform
        if self.augment:
            waveform = self._augment_waveform(waveform)

        # Convert to mel spectrogram
        mel_spec = self.mel_transform(waveform)        
        mel_spec = self.amplitude_to_db(mel_spec)      

        # Normalize
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-6)

        # SpecAugment
        if self.augment:
            mel_spec = self._spec_augment(mel_spec)

        return mel_spec.squeeze(0), label  

In [ ]:
train_dataset = AudioDataset(train_df, AUDIO_DIR, SAMPLE_RATE, N_MELS,
                             N_FFT, HOP_LENGTH, AUDIO_DURATION, augment=True)
val_dataset = AudioDataset(val_df, AUDIO_DIR, SAMPLE_RATE, N_MELS,
                           N_FFT, HOP_LENGTH, AUDIO_DURATION, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)


sample_mel, sample_label = train_dataset[0]
print(f"Mel spectrogram shape: {sample_mel.shape}")  
print(f"Label: {sample_label} ({IDX_TO_GENRE[sample_label]})")

In [12]:
class CRNN(nn.Module):
    """
    CNN encodes frequency features at each time step,
    then bidirectional GRU captures temporal dynamics.

    Input shape:  (batch, n_mels, time_steps)
    Output shape: (batch, num_classes)
    """
    def __init__(self, n_mels=128, cnn_features=128,
                 rnn_hidden=128, rnn_layers=2, num_classes=10, dropout=0.3):
        super().__init__()

        
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1)),  
            nn.Dropout2d(0.2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1)), 
            nn.Dropout2d(0.2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1)),  
            nn.Dropout2d(0.2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1)),
            nn.Dropout2d(0.2),
        )

        # After CNN: freq dimension is reduced by 2^4 = 16
        # n_mels=128 → 128/16 = 8
        cnn_out_freq = n_mels // 16
        self.cnn_to_rnn = nn.Linear(256 * cnn_out_freq, cnn_features)
        self.ln = nn.LayerNorm(cnn_features)

        # RNN: bidirectional GRU 
        self.rnn = nn.GRU(
            input_size=cnn_features,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if rnn_layers > 1 else 0
        )
        # Attention 
        self.attention = nn.Sequential(
            nn.Linear(rnn_hidden * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(rnn_hidden * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        
        batch_size = x.size(0)

        
        x = x.unsqueeze(1)

        # CNN: (batch, 256, n_mels//16, time_steps)
        x = self.cnn(x)

       
        x = x.permute(0, 3, 1, 2)        
        x = x.reshape(batch_size, x.size(1), -1)  

        
        x = self.cnn_to_rnn(x)            
        x = self.ln(x)

        
        x, _ = self.rnn(x)

        # Attention pooling
        attn_weights = self.attention(x)   #
        attn_weights = torch.softmax(attn_weights, dim=1)
        x = (x * attn_weights).sum(dim=1) 

        
        x = self.classifier(x)            
        return x

In [13]:
model = CRNN(
    n_mels=N_MELS,
    cnn_features=CNN_FEATURES,
    rnn_hidden=RNN_HIDDEN,
    rnn_layers=RNN_LAYERS,
    num_classes=NUM_CLASSES,
    dropout=RNN_DROPOUT
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nCRNN model parameters: {total_params:,}")


CRNN model parameters: 1,390,827


In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    name="model2-crnn-bigru-attention",
    config={
        "model": "CRNN",
        "sample_rate": SAMPLE_RATE,
        "n_mels": N_MELS,
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "audio_duration": AUDIO_DURATION,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "num_classes": NUM_CLASSES,
        "seed": SEED,
        "fold": TRAIN_FOLD,
        "cnn_features": CNN_FEATURES,
        "rnn_hidden": RNN_HIDDEN,
        "rnn_layers": RNN_LAYERS,
        "rnn_dropout": RNN_DROPOUT,
        "optimizer": "AdamW",
        "scheduler": "OneCycleLR",
        "augmentations": "VolumPerturb+NoiseInject+TimeShift+SpecAugment",
    }
)
wandb.watch(model, log="gradients", log_freq=50)


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS,
    steps_per_epoch=len(train_loader), pct_start=0.1
)

best_f1 = 0.0
best_model_path = os.path.join(OUTPUT_DIR, "model2_crnn_best.pth")
train_losses, val_losses = [], []
train_f1s, val_f1s = [], []

In [ ]:
print("\n" + "="*60)
print("TRAINING MODEL 2: CRNN (CNN + Bidirectional GRU)")
print("="*60)

for epoch in range(EPOCHS):
    # --- TRAIN ---
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for batch_idx, (mel_specs, labels) in enumerate(train_loader):
        mel_specs, labels = mel_specs.to(DEVICE), labels.to(DEVICE)

        outputs = model(mel_specs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    train_loss = running_loss / len(train_loader)
    train_f1 = f1_score(all_labels, all_preds, average='macro')
    train_losses.append(train_loss)
    train_f1s.append(train_f1)

    # --- VALIDATE ---
    model.eval()
    val_loss = 0.0
    all_val_preds, all_val_labels = [], []

    with torch.no_grad():
        for mel_specs, labels in val_loader:
            mel_specs, labels = mel_specs.to(DEVICE), labels.to(DEVICE)
            outputs = model(mel_specs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = outputs.argmax(dim=1).cpu().numpy()
            all_val_preds.extend(preds)
            all_val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')
    val_losses.append(val_loss)
    val_f1s.append(val_f1)

    lr_now = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | "
          f"Val Loss: {val_loss:.4f} F1: {val_f1:.4f} | "
          f"LR: {lr_now:.6f}")

    
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "train/macro_f1": train_f1,
        "val/loss": val_loss,
        "val/macro_f1": val_f1,
        "lr": lr_now,
        "best_val_f1": max(best_f1, val_f1),
    })

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_f1': val_f1,
        }, best_model_path)
        print(f"  -> Saved best model (F1: {val_f1:.4f})")

print(f"\n{'='*60}")
print(f"Best Validation Macro F1: {best_f1:.4f}")


In [ ]:
wandb.finish()
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_val_preds, all_val_labels = [], []
with torch.no_grad():
    for mel_specs, labels in val_loader:
        mel_specs = mel_specs.to(DEVICE)
        outputs = model(mel_specs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_val_preds.extend(preds)
        all_val_labels.extend(labels.numpy())

report_str = classification_report(all_val_labels, all_val_preds, target_names=GENRES)
print("\nClassification Report:")
print(report_str)

In [ ]:
print("\nModel 2 training complete!")
print(f"Best model saved to: {best_model_path}")

In [19]:
print("\n" + "="*60)
print("INFERENCE: Predicting test mashups")
print("="*60)

TEST_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"

best_model_path = "/kaggle/input/models/sreekaranreddy2005/model2-crnn/pytorch/default/1/model2_crnn_best.pth"
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model (Val F1: {checkpoint['val_f1']:.4f})")


infer_mel_transform = AT.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, power=2.0
).to(DEVICE)
infer_amp_to_db = AT.AmplitudeToDB(stype='power', top_db=80).to(DEVICE)

def predict_audio_crnn(filepath):
    """Load a test wav, chunk it, compute mel on GPU, predict with CRNN."""
    waveform, sr = torchaudio.load(filepath)

    
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    
    if sr != SAMPLE_RATE:
        resampler = AT.Resample(sr, SAMPLE_RATE)
        waveform = resampler(waveform)

    
    chunk_samples = SAMPLE_RATE * AUDIO_DURATION
    chunk_probs = []

    for start in range(0, waveform.shape[1], chunk_samples):
        chunk = waveform[:, start:start + chunk_samples]

        
        if chunk.shape[1] < chunk_samples // 2:
            continue

        
        if chunk.shape[1] < chunk_samples:
            chunk = torch.nn.functional.pad(chunk, (0, chunk_samples - chunk.shape[1]))

        
        chunk_gpu = chunk.to(DEVICE)
        with torch.no_grad():
            mel_spec = infer_mel_transform(chunk_gpu)
            mel_spec = infer_amp_to_db(mel_spec)
            mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-6)

            
            mel_input = mel_spec.squeeze(0)  
            logits = model(mel_input.unsqueeze(0))  
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            chunk_probs.append(probs[0])

    if len(chunk_probs) == 0:
        return np.ones(NUM_CLASSES) / NUM_CLASSES  

    return np.mean(chunk_probs, axis=0)


test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.wav')])
print(f"Found {len(test_files)} test songs")

# Predict
results = []
for i, filename in enumerate(test_files):
    filepath = os.path.join(TEST_DIR, filename)
    avg_probs = predict_audio_crnn(filepath)

    predicted_idx = np.argmax(avg_probs)
    predicted_genre = IDX_TO_GENRE[predicted_idx]
    song_id = int(os.path.splitext(filename)[0].replace('song', ''))

    results.append({'id': song_id, 'genre': predicted_genre})

    if (i + 1) % 100 == 0 or (i + 1) == len(test_files):
        print(f"  Processed {i+1}/{len(test_files)} | {filename} → {predicted_genre}")

results_df = pd.DataFrame(results)

results_df.to_csv("submission.csv", index=False)
print(f"\nSubmission saved to: submission.csv")



INFERENCE: Predicting test mashups
Loaded best model (Val F1: 0.7603)
Found 3020 test songs
  Processed 100/3020 | song0100.wav → country
  Processed 200/3020 | song0200.wav → country
  Processed 300/3020 | song0300.wav → hiphop
  Processed 400/3020 | song0400.wav → metal
  Processed 500/3020 | song0500.wav → pop
  Processed 600/3020 | song0600.wav → hiphop
  Processed 700/3020 | song0700.wav → pop
  Processed 800/3020 | song0800.wav → pop
  Processed 900/3020 | song0900.wav → jazz
  Processed 1000/3020 | song1000.wav → hiphop
  Processed 1100/3020 | song1100.wav → hiphop
  Processed 1200/3020 | song1200.wav → country
  Processed 1300/3020 | song1300.wav → country
  Processed 1400/3020 | song1400.wav → classical
  Processed 1500/3020 | song1500.wav → country
  Processed 1600/3020 | song1600.wav → reggae
  Processed 1700/3020 | song1700.wav → hiphop
  Processed 1800/3020 | song1800.wav → hiphop
  Processed 1900/3020 | song1900.wav → metal
  Processed 2000/3020 | song2000.wav → hiphop
 